In [3]:
import pandas as pd
import numpy as np
import cv2
import os
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.model_selection import train_test_split
import pickle
import time

In [4]:
# read metadata
path = "./data/CheXpert"

df = pd.read_csv(os.path.join(path, "train.csv"))

In [5]:
# df["Patient"] = df["Path"].apply(lambda x: x[26:38])
# Robust extraction of Patient ID (e.g., "patient00001")
df["Patient"] = df["Path"].apply(lambda x: str(x).split("/")[2] if len(str(x).split("/")) > 2 else "")
df

,Path,Sex,Age,Frontal/Lateral,AP/PA,No Finding,Enlarged Cardiomediastinum,Cardiomegaly,Lung Opacity,Lung Lesion,Edema,Consolidation,Pneumonia,Atelectasis,Pneumothorax,Pleural Effusion,Pleural Other,Fracture,Support Devices,Patient
0,CheXpert-v1.0-small/train/patient00001/study1/...,Female,68,Frontal,AP,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,1.0,patient00001
1,CheXpert-v1.0-small/train/patient00002/study2/...,Female,87,Frontal,AP,NaN,NaN,-1.0,1.0,NaN,-1.0,-1.0,NaN,-1.0,NaN,-1.0,NaN,1.0,NaN,patient00002
2,CheXpert-v1.0-small/train/patient00002/study1/...,Female,83,Frontal,AP,NaN,NaN,NaN,1.0,NaN,NaN,-1.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN,patient00002
3,CheXpert-v1.0-small/train/patient00002/study1/...,Female,83,Lateral,NaN,NaN,NaN,NaN,1.0,NaN,NaN,-1.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN,patient00002
4,CheXpert-v1.0-small/train/patient00003/study1/...,Male,41,Frontal,AP,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,patient00003
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
223409,CheXpert-v1.0-small/train/patient64537/study2/...,Male,59,Frontal,AP,NaN,NaN,NaN,-1.0,NaN,NaN,NaN,NaN,-1.0,0.0,1.0,NaN,NaN,NaN,patient64537
223410,CheXpert-v1.0-small/train/patient64537/study1/...,Male,59,Frontal,AP,NaN,NaN,NaN,-1.0,NaN,NaN,NaN,0.0,-1.0,NaN,-1.0,NaN,NaN,NaN,patient64537
223411,CheXpert-v1.0-small/train/patient64538/study1/...,Female,0,Frontal,AP,NaN,NaN,NaN,NaN,NaN,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,patient64538
223412,CheXpert-v1.0-small/train/patient64539/study1/...,Female,0,Frontal,AP,NaN,NaN,1.0,1.0,NaN,NaN,NaN,-1.0,1.0,0.0,NaN,NaN,NaN,0.0,patient64539


In [6]:
# df_race = pd.read_excel(os.path.join(path, "CHEXPERT DEMO.xlsx"), sheet_name="CheXpert Demo 11-13")

# df = df.merge(df_race, how="left", left_on="Patient", right_on="PATIENT")
# df["Race"] = df["PRIMARY_RACE"]
# df = df.loc[df["PATIENT"].notna() & df["Patient"].notna()].drop(
#     columns=["Patient", "PATIENT", "ETHNICITY", "AGE_AT_CXR", "PRIMARY_RACE"]
# )

# df



In [49]:
df = df[df["Sex"].isin(df["GENDER"])].drop(columns="GENDER")
df

,Path,Sex,Age,Frontal/Lateral,AP/PA,No Finding,Enlarged Cardiomediastinum,Cardiomegaly,Lung Opacity,Lung Lesion,Edema,Consolidation,Pneumonia,Atelectasis,Pneumothorax,Pleural Effusion,Pleural Other,Fracture,Support Devices,Race
0,CheXpert-v1.0-small/train/patient00001/study1/...,Female,68,Frontal,AP,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,1.0,Other
1,CheXpert-v1.0-small/train/patient00002/study2/...,Female,87,Frontal,AP,NaN,NaN,-1.0,1.0,NaN,-1.0,-1.0,NaN,-1.0,NaN,-1.0,NaN,1.0,NaN,"White, non-Hispanic"
2,CheXpert-v1.0-small/train/patient00002/study1/...,Female,83,Frontal,AP,NaN,NaN,NaN,1.0,NaN,NaN,-1.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN,"White, non-Hispanic"
3,CheXpert-v1.0-small/train/patient00002/study1/...,Female,83,Lateral,NaN,NaN,NaN,NaN,1.0,NaN,NaN,-1.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN,"White, non-Hispanic"
4,CheXpert-v1.0-small/train/patient00003/study1/...,Male,41,Frontal,AP,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,"White, non-Hispanic"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
223406,CheXpert-v1.0-small/train/patient64535/study1/...,Male,60,Frontal,AP,NaN,NaN,NaN,1.0,NaN,NaN,-1.0,NaN,-1.0,NaN,NaN,NaN,NaN,NaN,Black or African American
223407,CheXpert-v1.0-small/train/patient64536/study2/...,Female,61,Frontal,AP,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,Other
223408,CheXpert-v1.0-small/train/patient64536/study1/...,Female,61,Frontal,AP,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,1.0,NaN,NaN,NaN,NaN,1.0,Other
223409,CheXpert-v1.0-small/train/patient64537/study2/...,Male,59,Frontal,AP,NaN,NaN,NaN,-1.0,NaN,NaN,NaN,NaN,-1.0,0.0,1.0,NaN,NaN,NaN,Black or African American


In [50]:
# remove age/sex/race == null
df = df[~df["Sex"].isnull()]
df = df[~df["Age"].isnull()]
df = df[~df["Race"].isnull()]
df

,Path,Sex,Age,Frontal/Lateral,AP/PA,No Finding,Enlarged Cardiomediastinum,Cardiomegaly,Lung Opacity,Lung Lesion,Edema,Consolidation,Pneumonia,Atelectasis,Pneumothorax,Pleural Effusion,Pleural Other,Fracture,Support Devices,Race
0,CheXpert-v1.0-small/train/patient00001/study1/...,Female,68,Frontal,AP,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,1.0,Other
1,CheXpert-v1.0-small/train/patient00002/study2/...,Female,87,Frontal,AP,NaN,NaN,-1.0,1.0,NaN,-1.0,-1.0,NaN,-1.0,NaN,-1.0,NaN,1.0,NaN,"White, non-Hispanic"
2,CheXpert-v1.0-small/train/patient00002/study1/...,Female,83,Frontal,AP,NaN,NaN,NaN,1.0,NaN,NaN,-1.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN,"White, non-Hispanic"
3,CheXpert-v1.0-small/train/patient00002/study1/...,Female,83,Lateral,NaN,NaN,NaN,NaN,1.0,NaN,NaN,-1.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN,"White, non-Hispanic"
4,CheXpert-v1.0-small/train/patient00003/study1/...,Male,41,Frontal,AP,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,"White, non-Hispanic"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
223406,CheXpert-v1.0-small/train/patient64535/study1/...,Male,60,Frontal,AP,NaN,NaN,NaN,1.0,NaN,NaN,-1.0,NaN,-1.0,NaN,NaN,NaN,NaN,NaN,Black or African American
223407,CheXpert-v1.0-small/train/patient64536/study2/...,Female,61,Frontal,AP,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,Other
223408,CheXpert-v1.0-small/train/patient64536/study1/...,Female,61,Frontal,AP,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,1.0,NaN,NaN,NaN,NaN,1.0,Other
223409,CheXpert-v1.0-small/train/patient64537/study2/...,Male,59,Frontal,AP,NaN,NaN,NaN,-1.0,NaN,NaN,NaN,NaN,-1.0,0.0,1.0,NaN,NaN,NaN,Black or African American


In [51]:
df["Race"].value_counts()

Race
White                                        102402
Other                                         28095
White, non-Hispanic                           22154
Asian                                         20434
Unknown                                       15186
Black or African American                      9909
Race and Ethnicity Unknown                     8716
Other, Hispanic                                3621
Native Hawaiian or Other Pacific Islander      2809
Asian, non-Hispanic                            2793
Black, non-Hispanic                            2000
White, Hispanic                                 922
Other, non-Hispanic                             566
American Indian or Alaska Native                457
Patient Refused                                 405
Pacific Islander, non-Hispanic                  337
Native American, non-Hispanic                    55
Black, Hispanic                                  52
Asian, Hispanic                                  37
Native 

In [52]:
mask = df["Race"].str.contains("White", na=False)
df.loc[mask, "Race"] = "White"
df.loc[~mask, "Race"] = "Non-White"

df["No Finding"].replace(-1, 0, inplace=True)
df["No Finding"].fillna(0, inplace=True)

In [53]:
df["Race"].value_counts()

Race
White        125491
Non-White     95515
Name: count, dtype: int64

In [54]:
df["Sex"].value_counts()

Sex
Male      131155
Female     89851
Name: count, dtype: int64

In [55]:
df["No Finding"].value_counts()

No Finding
0.0    198856
1.0     22150
Name: count, dtype: int64

In [56]:
df.to_csv(os.path.join(path, "test.csv"), index=False)

In [59]:
# sex

tem_train = pd.read_csv("./data/CheXpert/new_train.csv")
tem_test = pd.read_csv("./data/CheXpert/new_test.csv")

df = pd.read_csv(os.path.join(path, "all.csv"))
df_train = df.loc[df["Path"].isin(tem_train["Path"])].reset_index(drop=True)
df_test = df.loc[df["Path"].isin(tem_test["Path"])].reset_index(drop=True)

# assert len(df_train) == len(tem_train), f"{len(df_train)}, {len(tem_train)}"

df_train.to_csv(os.path.join(path, "train.csv"), index=False)
df_test.to_csv(os.path.join(path, "test.csv"), index=False)

In [13]:
# age

tem_test = pd.read_csv("./data/CheXpert/cxp_test_age.csv")

df = pd.read_csv(os.path.join(path, "all.csv"))
df_test = df.loc[df["Path"].isin(tem_test["Path"])].reset_index(drop=True)

# assert len(df_train) == len(tem_train), f"{len(df_train)}, {len(tem_train)}"

print(len(df_test))
df_test.to_csv(os.path.join(path, "test_age.csv"), index=False)

tem_test["Age_binary"].value_counts()

33757


Age_binary
0    17081
1    17081
Name: count, dtype: int64

In [10]:
df_test["Age"].value_counts()

Age
90    1128
57     797
58     796
61     753
56     750
      ... 
20     199
21     194
26     187
19     182
18      99
Name: count, Length: 73, dtype: int64

In [6]:
# race

tem_test = pd.read_csv("./data/CheXpert/cxp_test_race.csv")

df = pd.read_csv(os.path.join(path, "all.csv"))
df_test = df.loc[df["Path"].isin(tem_test["Path"])].reset_index(drop=True)

# assert len(df_train) == len(tem_train), f"{len(df_train)}, {len(tem_train)}"

df_test.to_csv(os.path.join(path, "test_race.csv"), index=False)

My preprocessing:

In [2]:
import os
import pandas as pd
import numpy as np

# Adjust this path if your folder is named differently
chexpert_root = "./data/CheXpert"
# If unzipped from kaggle directly into current folder, fallback check:
if not os.path.exists(chexpert_root):
    if os.path.exists("./CheXpert"):
        chexpert_root = "./CheXpert"
    elif os.path.exists("./CheXpert-v1.0-small"):
        chexpert_root = "./CheXpert-v1.0-small"
    else:
        chexpert_root = "."

print(f"=== Reading files from: {chexpert_root} ===")

# Look for standard CheXpert files
train_csv_path = os.path.join(chexpert_root, "train.csv")
valid_csv_path = os.path.join(chexpert_root, "valid.csv")
demo_path = os.path.join(chexpert_root, "CHEXPERT DEMO.xlsx")

if not os.path.exists(train_csv_path):
    # Search recursively if nested
    print("Searching for train.csv...")
    for root, dirs, files in os.walk(chexpert_root):
        if "train.csv" in files:
            train_csv_path = os.path.join(root, "train.csv")
            break

print(f"Loading metadata from: {train_csv_path}")
df = pd.read_csv(train_csv_path)

print("\n" + "="*50)
print(f"1. OVERVIEW & COLUMNS")
print("="*50)
print(f"Total Rows (Images): {len(df):,}")
print(f"Available Columns:\n{list(df.columns)}")

# Extract Patient ID
if "Path" in df.columns:
    df["Patient"] = df["Path"].apply(lambda x: str(x).split("/")[2] if len(str(x).split("/")) > 2 else "Unknown")
    print(f"Unique Patients: {df['Patient'].nunique():,}")

# Check Demographic Excel
if os.path.exists(demo_path):
    print(f"\nFound Demographic File: {demo_path}")
    try:
        df_demo = pd.read_excel(demo_path)
        print("Demographic Columns:", list(df_demo.columns))
        # Merge to inspect race
        patient_col = [c for c in df_demo.columns if "PATIENT" in c.upper() or "PAT_ID" in c.upper()]
        if patient_col:
            df = df.merge(df_demo, how="left", left_on="Patient", right_on=patient_col[0])
            print("Successfully merged demographic data!")
    except Exception as e:
        print(f"Could not read demographic excel: {e}")
else:
    print(f"\n[Note] 'CHEXPERT DEMO.xlsx' not found in root. If race is needed, we will check if it is already inside train.csv.")

print("\n" + "="*50)
print("2. PROTECTED ATTRIBUTES BREAKDOWN")
print("="*50)

# 1. Sex
if "Sex" in df.columns:
    print("\n--- Sex Distribution ---")
    print(df["Sex"].value_counts(dropna=False))

# 2. Age
if "Age" in df.columns:
    print("\n--- Age Summary ---")
    print(df["Age"].describe())
    
    # Group into clinical brackets
    bins = [0, 40, 60, 80, 120]
    labels = ["<40", "40-59", "60-79", "80+"]
    df["Age_Bracket"] = pd.cut(df["Age"], bins=bins, labels=labels, right=False)
    print("\n--- Age Group Distribution ---")
    print(df["Age_Bracket"].value_counts(dropna=False).sort_index())

# 3. Race (if present in train.csv or demo)
race_cols = [c for c in df.columns if "RACE" in c.upper() or "ETHNICITY" in c.upper()]
if race_cols:
    for rc in race_cols:
        print(f"\n--- Race / Ethnicity Breakdown ({rc}) ---")
        print(df[rc].value_counts(dropna=False).head(10))
else:
    print("\n--- Race: No race column detected in metadata ---")

print("\n" + "="*50)
print("3. PATHOLOGY / TARGET CLASS BREAKDOWN")
print("="*50)

# Typical CheXpert target conditions
findings = [
    "No Finding", "Enlarged Cardiomediastinum", "Cardiomegaly", 
    "Lung Opacity", "Lung Lesion", "Edema", "Consolidation", 
    "Pneumonia", "Atelectasis", "Pneumothorax", "Pleural Effusion", 
    "Pleural Other", "Fracture", "Support Devices"
]

available_findings = [f for f in findings if f in df.columns]

if available_findings:
    summary = []
    for f in available_findings:
        counts = df[f].value_counts(dropna=False)
        pos = counts.get(1.0, 0)
        neg = counts.get(0.0, 0)
        unc = counts.get(-1.0, 0)
        nan = df[f].isna().sum()
        summary.append({
            "Pathology": f,
            "Positive (1)": pos,
            "Negative (0)": neg,
            "Uncertain (-1)": unc,
            "Blank (NaN)": nan
        })
    summary_df = pd.DataFrame(summary)
    print(summary_df.to_string(index=False))
else:
    print("Standard finding columns not found. Printing value counts for all numeric columns:")
    for col in df.select_dtypes(include=[np.number]).columns:
        if col not in ["Age"]:
            print(f"\n--- {col} ---")
            print(df[col].value_counts(dropna=False).head(5))

print("\n" + "="*50)
print("Inspection complete.")
print("="*50)

=== Reading files from: ./data/CheXpert ===
Loading metadata from: ./data/CheXpert/train.csv

1. OVERVIEW & COLUMNS
Total Rows (Images): 223,414
Available Columns:
['Path', 'Sex', 'Age', 'Frontal/Lateral', 'AP/PA', 'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices']
Unique Patients: 64,540

[Note] 'CHEXPERT DEMO.xlsx' not found in root. If race is needed, we will check if it is already inside train.csv.

2. PROTECTED ATTRIBUTES BREAKDOWN

--- Sex Distribution ---
Sex
Male       132636
Female      90777
Unknown         1
Name: count, dtype: int64

--- Age Summary ---
count    223414.000000
mean         60.430653
std          17.820925
min           0.000000
25%          49.000000
50%          62.000000
75%          74.000000
max          90.000000
Name: Age, dtype: float64

--- Age Group Distribution ---
Age_Brac

In [3]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

# Root directory where CheXpert files are stored
base_dir = "./data/CheXpert"
train_csv_path = os.path.join(base_dir, "train.csv")

print(f"Loading raw metadata from: {train_csv_path}")
df = pd.read_csv(train_csv_path)

print(f"Initial raw samples: {len(df):,}")

Loading raw metadata from: ./data/CheXpert/train.csv
Initial raw samples: 223,414


In [4]:
# 1. Extract Patient ID for grouped splitting (e.g., "patient00001")
df["Patient"] = df["Path"].apply(lambda x: str(x).split("/")[2] if len(str(x).split("/")) > 2 else "unknown")

# 2. Clean Sex attribute (removes 1 unknown row)
df = df[df["Sex"].isin(["Male", "Female"])].copy()

# 3. Clean Age attribute (removes invalid ages <= 0 or NaN)
df = df[df["Age"].notna() & (df["Age"] > 0)].copy()

# 4. Create Age Groups (Clinical life stages for fairness metrics)
df["Age_Group"] = pd.qcut(df["Age"], q=4, labels=["Q1 (<49)", "Q2 (49-61)", "Q3 (62-73)", "Q4 (74+)"])
# df["Age_Group"] = df["Age"].apply(categorize_age)

print(f"Cleaned samples: {len(df):,}")
print(f"Unique patients: {df['Patient'].nunique():,}")
print("\n--- Sex Distribution ---")
print(df["Sex"].value_counts())
print("\n--- Age Group Distribution ---")
print(df["Age_Group"].value_counts())

Cleaned samples: 223,410
Unique patients: 64,536

--- Sex Distribution ---
Sex
Male      132636
Female     90774
Name: count, dtype: int64

--- Age Group Distribution ---
Age_Group
Q2 (49-61)    58380
Q1 (<49)      57200
Q4 (74+)      54453
Q3 (62-73)    53377
Name: count, dtype: int64


In [5]:
# 1. Clean and binarize 'No Finding' directly in-place
# 0 = Diseased / Finding present, 1 = Normal / No Finding
df["No Finding"] = df["No Finding"].fillna(0).replace(-1, 0).astype(int)

# Optional: also assign to 'label' if any helper script checks for it
df["label"] = df["No Finding"]

# 2. Normalize Path prefix for DataLoader
df["Path"] = df["Path"].apply(lambda x: x.replace("CheXpert-v1.0-small/", "") if "CheXpert-v1.0-small/" in str(x) else str(x))

print("\n--- Target Label Distribution (No Finding) ---")
print(df["No Finding"].value_counts().rename(index={0: "Diseased (0)", 1: "Normal (1)"}))


--- Target Label Distribution (No Finding) ---
No Finding
Diseased (0)    201031
Normal (1)       22379
Name: count, dtype: int64


In [6]:

# 70% Train, 10% Validation, 20% Test (strictly split on 'Patient' ID)
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_val_idx, test_idx = next(gss_test.split(df, groups=df["Patient"]))

df_train_val = df.iloc[train_val_idx].reset_index(drop=True)
df_test = df.iloc[test_idx].reset_index(drop=True)

gss_val = GroupShuffleSplit(n_splits=1, test_size=0.125, random_state=42) # 0.125 * 0.80 = 10% total
train_idx, val_idx = next(gss_val.split(df_train_val, groups=df_train_val["Patient"]))

df_train = df_train_val.iloc[train_idx].reset_index(drop=True)
df_val = df_train_val.iloc[val_idx].reset_index(drop=True)

print(f"Train set: {len(df_train):,} images ({df_train['Patient'].nunique():,} unique patients)")
print(f"Val set:   {len(df_val):,} images ({df_val['Patient'].nunique():,} unique patients)")
print(f"Test set:  {len(df_test):,} images ({df_test['Patient'].nunique():,} unique patients)")

# Verification: Assert zero patient overlap
train_p = set(df_train["Patient"])
val_p = set(df_val["Patient"])
test_p = set(df_test["Patient"])
assert len(train_p.intersection(test_p)) == 0, "Patient leakage detected between Train and Test!"
assert len(val_p.intersection(test_p)) == 0, "Patient leakage detected between Val and Test!"
print("\nValidation passed: 0% patient leakage across all splits.")

output_dir = base_dir

# 1. Core dataset splits
df_train.to_csv(os.path.join(output_dir, "train_processed.csv"), index=False)
df_val.to_csv(os.path.join(output_dir, "val_processed.csv"), index=False)
df_test.to_csv(os.path.join(output_dir, "test_processed.csv"), index=False)

# 2. Sensitive attribute specific evaluation files expected by FairMedFM
df_test.to_csv(os.path.join(output_dir, "test_sex.csv"), index=False)
df_test.to_csv(os.path.join(output_dir, "test_age.csv"), index=False)

print(f"Successfully saved all processed CSV files to: {output_dir}")

Train set: 156,516 images (45,174 unique patients)
Val set:   22,637 images (6,454 unique patients)
Test set:  44,257 images (12,908 unique patients)

Validation passed: 0% patient leakage across all splits.
Successfully saved all processed CSV files to: ./data/CheXpert


In [2]:
import pandas as pd

# Update these paths to where your CheXpert CSVs are located
path_train = './data/CheXpert/train_processed.csv'
path_val = './data/CheXpert/val_processed.csv'
path_test = './data/CheXpert/test_processed.csv'


# Load the data
df_train = pd.read_csv(path_train)
df_val = pd.read_csv(path_val)
df_test = pd.read_csv(path_test)

# 1. Total Sizes, Splits, and Patients
total_images = len(df_train) + len(df_val) + len(df_test)
if 'Patient' in df_train.columns:
    total_patients = df_train['Patient'].nunique() + df_val['Patient'].nunique() + df_test['Patient'].nunique()
else:
    total_patients = "Unknown"

print(f"--- 1. DATASET SIZES ---")
print(f"Total Images: {total_images:,} (from {total_patients} unique patients)")
print(f"Train: {len(df_train):,} | Validation: {len(df_val):,} | Test: {len(df_test):,}")

# 2. Clinical Task (No Finding)
print("\n--- 2. CLINICAL TASK (Train Set) ---")
print("Target Column: 'No Finding' (1 = Normal/No Finding, 0 = Diseased)")
prevalence = df_train['No Finding'].value_counts(normalize=True) * 100
print(prevalence.round(2).to_string())

# 3 & 4. Demographic Attributes and Categories
print("\n--- 3 & 4. DEMOGRAPHICS (Test Set) ---")
if 'Sex' in df_test.columns:
    print("\nSex Categories:")
    print(df_test['Sex'].value_counts())

if 'Age_Group' in df_test.columns:
    print("\nAge Group Categories:")
    print(df_test['Age_Group'].value_counts())
elif 'Age' in df_test.columns:
    print("\nNumeric Age detected! You need to discretize it.")

--- 1. DATASET SIZES ---
Total Images: 223,410 (from 64536 unique patients)
Train: 156,516 | Validation: 22,637 | Test: 44,257

--- 2. CLINICAL TASK (Train Set) ---
Target Column: 'No Finding' (1 = Normal/No Finding, 0 = Diseased)
No Finding
0    90.01
1     9.99

--- 3 & 4. DEMOGRAPHICS (Test Set) ---

Sex Categories:
Sex
Male      26043
Female    18214
Name: count, dtype: int64

Age Group Categories:
Age_Group
Q2 (49-61)    11673
Q1 (<49)      11137
Q4 (74+)      10985
Q3 (62-73)    10462
Name: count, dtype: int64
